# Posture Detection V2 — 11 Webcam Domain-Mismatch Diagnostics

Records a short, labelled five-posture webcam sample and diagnoses why the Notebook 09 public classifier rejects live landmarks as out of distribution (OOD). It compares coordinate adapters **without modifying or overwriting the production model**.

The capture remains local. Press **Q** at any time to stop safely.


## 1. Load the Notebook 09 contract, model, public reference, and MediaPipe

Keep this notebook in `project_v2` or `project_v2/notebooks`. Notebook 09 must already have produced the 57-feature model and contract.


In [ ]:
# Uncomment if needed: %pip install -q mediapipe opencv-python scikit-learn joblib seaborn
import json, time
from datetime import datetime
from pathlib import Path

import cv2
import joblib
import matplotlib.pyplot as plt
import mediapipe as mp
import numpy as np
import pandas as pd
import seaborn as sns

wd=Path.cwd()
PROJECT_ROOT=wd.parent if wd.name=='notebooks' else wd if wd.name=='project_v2' else wd/'project_v2'
MODEL_FILE=PROJECT_ROOT/'models'/'multiposture_upper57_rf.joblib'
CONTRACT_FILE=PROJECT_ROOT/'models'/'multiposture_upper57_contract.json'
PUBLIC_CSV=PROJECT_ROOT/'data'/'public_multiposture'/'data.csv'
MODEL_TASK=PROJECT_ROOT/'models'/'pose_landmarker_full.task'
OUTPUT_DIR=PROJECT_ROOT/'results'/'webcam_domain_diagnostics'
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
for p in (MODEL_FILE,CONTRACT_FILE,PUBLIC_CSV,MODEL_TASK):
    if not p.exists(): raise FileNotFoundError(f'Missing {p}. Complete Notebooks 06 and 09 first.')

rf=joblib.load(MODEL_FILE)
if hasattr(rf,'n_jobs'): rf.n_jobs=1
contract=json.loads(CONTRACT_FILE.read_text(encoding='utf-8'))
FEATURE_COLUMNS=list(contract['feature_columns'])
KEEP_INDICES=[int(i) for i in contract['landmark_indices']]
KEEP_NAMES=list(contract['landmark_names'])
MIN_PROB=float(contract.get('minimum_probability',.55))
MAX_OOD=float(contract.get('maximum_ood_fraction',.20))
public=pd.read_csv(PUBLIC_CSV,usecols=FEATURE_COLUMNS).astype(np.float32)
if len(FEATURE_COLUMNS)!=57 or int(getattr(rf,'n_features_in_',-1))!=57:
    raise ValueError('Expected the Notebook 09 57-feature artifacts.')
print('Public rows:',len(public),'Features:',len(FEATURE_COLUMNS),'Classes:',list(rf.classes_))
print(f'Acceptance thresholds: probability >= {MIN_PROB:.0%}, OOD <= {MAX_OOD:.0%}')


## 2. Controlled webcam capture

The notebook cycles through five prompts. Each has a **3-second preparation period** followed by **8 seconds of recording**:

1. upright;
2. lean forward;
3. lean backward;
4. lean left;
5. lean right.

Keep your hips and upper body visible. These prompts are diagnostic labels, not clinical ground truth.


In [ ]:
CAMERA_INDEX=0
PREP_SECONDS=3
RECORD_SECONDS=8
SAMPLE_EVERY=3
PHASES=['upright','lean_forward','lean_backward','lean_left','lean_right']

Base=mp.tasks.BaseOptions
Landmarker=mp.tasks.vision.PoseLandmarker
Options=mp.tasks.vision.PoseLandmarkerOptions
Mode=mp.tasks.vision.RunningMode
opts=Options(base_options=Base(model_asset_path=str(MODEL_TASK)),running_mode=Mode.VIDEO,num_poses=1,
             min_pose_detection_confidence=.5,min_pose_presence_confidence=.5,min_tracking_confidence=.5)

def feature_row(landmarks):
    xyz=np.array([[p.x,p.y,p.z] for p in landmarks],dtype=np.float32)
    hip=(xyz[23]+xyz[24])/2
    xyz-=hip
    values={f'{name}_{axis}':float(xyz[i,j]) for i,name in zip(KEEP_INDICES,KEEP_NAMES) for j,axis in enumerate('xyz')}
    visibility=float(min(min(landmarks[i].visibility,getattr(landmarks[i],'presence',landmarks[i].visibility)) for i in KEEP_INDICES))
    return values,visibility

cap=cv2.VideoCapture(CAMERA_INDEX)
cap.set(cv2.CAP_PROP_FRAME_WIDTH,1280); cap.set(cv2.CAP_PROP_FRAME_HEIGHT,720)
if not cap.isOpened(): raise RuntimeError('Could not open webcam.')
records=[]; frame_id=0; stop=False; started=time.perf_counter()
with Landmarker.create_from_options(opts) as detector:
    try:
        for phase in PHASES:
            prep_start=time.perf_counter()
            while time.perf_counter()-prep_start<PREP_SECONDS:
                ok,frame=cap.read()
                if not ok: stop=True; break
                frame=cv2.flip(frame,1); remain=PREP_SECONDS-(time.perf_counter()-prep_start)
                cv2.putText(frame,f'GET READY: {phase.replace("_"," ").upper()}',(25,55),cv2.FONT_HERSHEY_SIMPLEX,1.0,(0,200,255),2)
                cv2.putText(frame,f'Starts in {max(1,int(np.ceil(remain)))}',(25,100),cv2.FONT_HERSHEY_SIMPLEX,.8,(255,255,255),2)
                cv2.imshow('Notebook 11 Domain Diagnostic',frame)
                if cv2.waitKey(1)&0xFF==ord('q'): stop=True; break
            if stop: break
            phase_start=time.perf_counter()
            while time.perf_counter()-phase_start<RECORD_SECONDS:
                ok,frame=cap.read()
                if not ok: stop=True; break
                frame=cv2.flip(frame,1); now=time.perf_counter(); ts=int((now-started)*1000)
                result=detector.detect_for_video(mp.Image(image_format=mp.ImageFormat.SRGB,data=cv2.cvtColor(frame,cv2.COLOR_BGR2RGB)),ts)
                detected=bool(result.pose_landmarks)
                if detected and frame_id%SAMPLE_EVERY==0:
                    values,visibility=feature_row(result.pose_landmarks[0])
                    records.append({'phase':phase,'seconds':now-started,'minimum_visibility':visibility,**values})
                remain=RECORD_SECONDS-(now-phase_start)
                color=(0,220,0) if detected else (0,0,255)
                cv2.putText(frame,f'RECORD: {phase.replace("_"," ").upper()}',(25,55),cv2.FONT_HERSHEY_SIMPLEX,1.0,color,2)
                cv2.putText(frame,f'{remain:.1f}s  samples: {len(records)}',(25,100),cv2.FONT_HERSHEY_SIMPLEX,.75,(255,255,255),2)
                cv2.putText(frame,'Q: stop',(25,frame.shape[0]-25),cv2.FONT_HERSHEY_SIMPLEX,.6,(255,255,255),1)
                cv2.imshow('Notebook 11 Domain Diagnostic',frame)
                if cv2.waitKey(1)&0xFF==ord('q'): stop=True; break
                frame_id+=1
            if stop: break
    finally:
        cap.release(); cv2.destroyAllWindows()

captured=pd.DataFrame(records)
if captured.empty: raise RuntimeError('No pose samples were captured. Check camera visibility and run this cell again.')
stamp=datetime.now().strftime('%Y%m%d_%H%M%S')
CAPTURE_FILE=OUTPUT_DIR/f'webcam57_capture_{stamp}.csv'
captured.to_csv(CAPTURE_FILE,index=False)
print('Saved:',CAPTURE_FILE.resolve())
display(captured.groupby('phase').agg(samples=('phase','size'),median_visibility=('minimum_visibility','median')))


## 3. Compare coordinate adapters

Each candidate is evaluated against equivalently transformed public coordinates. `z_flip` and `mirror_swap` deliberately alter only webcam coordinates to test convention mismatches. The safest candidate has low median OOD and a high percentage of frames below the 20% OOD threshold.


In [ ]:
webcam=captured[FEATURE_COLUMNS].astype(np.float32).copy()

def as_array(df):
    return df[FEATURE_COLUMNS].to_numpy(np.float32).reshape(-1,len(KEEP_NAMES),3)

name_to_pos={name:i for i,name in enumerate(KEEP_NAMES)}
LS,RS=name_to_pos['left_shoulder'],name_to_pos['right_shoulder']
LH,RH=name_to_pos['left_hip'],name_to_pos['right_hip']

def normalize(df,mode):
    a=as_array(df).copy()
    if mode in ('torso','shoulder'):
        if mode=='torso':
            shoulder_mid=(a[:,LS]+a[:,RS])/2; hip_mid=(a[:,LH]+a[:,RH])/2
            scale=np.linalg.norm(shoulder_mid-hip_mid,axis=1)
        else:
            scale=np.linalg.norm(a[:,LS]-a[:,RS],axis=1)
        a/=np.maximum(scale,1e-6)[:,None,None]
    return pd.DataFrame(a.reshape(len(a),-1),columns=FEATURE_COLUMNS,index=df.index)

def webcam_z_flip(df):
    out=df.copy(); zcols=[c for c in FEATURE_COLUMNS if c.endswith('_z')]; out[zcols]*=-1; return out

def webcam_mirror_swap(df):
    a=as_array(df).copy(); a[:,:,0]*=-1
    for left in [n for n in KEEP_NAMES if n.startswith('left_')]:
        right='right_'+left[5:]
        if right in name_to_pos:
            li,ri=name_to_pos[left],name_to_pos[right]
            a[:,[li,ri]]=a[:,[ri,li]]
    return pd.DataFrame(a.reshape(len(a),-1),columns=FEATURE_COLUMNS,index=df.index)

candidates={
    'current_raw':(normalize(public,'raw'),normalize(webcam,'raw')),
    'torso_scaled':(normalize(public,'torso'),normalize(webcam,'torso')),
    'shoulder_scaled':(normalize(public,'shoulder'),normalize(webcam,'shoulder')),
    'torso_scaled_webcam_z_flip':(normalize(public,'torso'),webcam_z_flip(normalize(webcam,'torso'))),
    'torso_scaled_webcam_mirror_swap':(normalize(public,'torso'),webcam_mirror_swap(normalize(webcam,'torso'))),
}

rows=[]; ood_masks={}
for name,(reference,live) in candidates.items():
    lower=reference.quantile(.005); upper=reference.quantile(.995)
    mask=live.lt(lower,axis=1)|live.gt(upper,axis=1)
    fractions=mask.mean(axis=1)
    ood_masks[name]=mask
    rows.append({'adapter':name,'median_ood_fraction':float(fractions.median()),
                 'mean_ood_fraction':float(fractions.mean()),
                 'ood_pass_percent':float(fractions.le(MAX_OOD).mean()*100)})

adapter_summary=pd.DataFrame(rows).sort_values(['ood_pass_percent','median_ood_fraction'],ascending=[False,True])
display(adapter_summary)
adapter_summary.to_csv(OUTPUT_DIR/f'adapter_summary_{stamp}.csv',index=False)

phase_rows=[]
for name,mask in ood_masks.items():
    fractions=mask.mean(axis=1)
    temp=pd.DataFrame({'phase':captured['phase'].to_numpy(),'ood_fraction':fractions.to_numpy()})
    part=temp.groupby('phase')['ood_fraction'].agg(median_ood_fraction='median',ood_pass_percent=lambda s:s.le(MAX_OOD).mean()*100).reset_index()
    part.insert(0,'adapter',name); phase_rows.append(part)
phase_summary=pd.concat(phase_rows,ignore_index=True)
display(phase_summary.pivot(index='phase',columns='adapter',values='median_ood_fraction'))
phase_summary.to_csv(OUTPUT_DIR/f'phase_adapter_summary_{stamp}.csv',index=False)


## 4. Identify the failing coordinates and inspect raw classifier behavior

This section ranks features by how often they exceed the public 0.5–99.5% range. It also reports the current classifier's predictions, but does not treat prompted poses as precise ground-truth annotations.


In [ ]:
raw_mask=ood_masks['current_raw']
public_lower=public.quantile(.005); public_upper=public.quantile(.995)
top_ood=pd.DataFrame({
    'feature':FEATURE_COLUMNS,
    'out_of_range_percent':raw_mask.mean(axis=0).to_numpy()*100,
    'public_low':public_lower.to_numpy(),
    'public_high':public_upper.to_numpy(),
    'webcam_median':webcam.median().to_numpy(),
    'webcam_low':webcam.quantile(.005).to_numpy(),
    'webcam_high':webcam.quantile(.995).to_numpy(),
}).sort_values('out_of_range_percent',ascending=False)
display(top_ood.head(20))
top_ood.to_csv(OUTPUT_DIR/f'top_ood_features_{stamp}.csv',index=False)

probabilities=rf.predict_proba(webcam)
indices=np.argmax(probabilities,axis=1)
predicted=np.asarray(rf.classes_)[indices]
confidence=probabilities[np.arange(len(probabilities)),indices]
raw_fraction=raw_mask.mean(axis=1).to_numpy()
prediction_audit=pd.DataFrame({'prompted_phase':captured['phase'].to_numpy(),'predicted':predicted,
                               'confidence':confidence,'ood_fraction':raw_fraction,
                               'accepted':(confidence>=MIN_PROB)&(raw_fraction<=MAX_OOD)})
display(pd.crosstab(prediction_audit['prompted_phase'],prediction_audit['predicted']))
display(prediction_audit.groupby('prompted_phase').agg(samples=('predicted','size'),median_confidence=('confidence','median'),median_ood=('ood_fraction','median'),accepted_percent=('accepted',lambda s:s.mean()*100)))
prediction_audit.to_csv(OUTPUT_DIR/f'raw_prediction_audit_{stamp}.csv',index=False)

best=adapter_summary.iloc[0]
print('Best diagnostic adapter:',best['adapter'])
print(f"Median OOD: {best['median_ood_fraction']:.1%}; OOD pass: {best['ood_pass_percent']:.1f}%")
if best['ood_pass_percent']<70:
    print('RESULT: Coordinate adaptation alone is insufficient. Do not relax the production OOD gate.')
else:
    print('RESULT: This adapter is promising, but must be used to retrain and re-evaluate the classifier before webcam deployment.')
print('Reports saved in:',OUTPUT_DIR.resolve())


## What to send back

Send this executed notebook. We will use the adapter table and top-OOD feature table to choose one of three evidence-based next steps:

1. implement and retrain with a validated normalization adapter;
2. remove systematically incompatible coordinates and retrain again;
3. collect a small local calibration/adaptation set while retaining MotionBERT as the safe fallback.
